# Minimal GCN disease-gene prioritization

This notebook first illustrates the protocol for one disease, then trains one shared four-layer graph convolutional network across **all diseases** on the complete loaded PPI graph. For each disease task, known genes are split into visible seeds, positive training targets, and held-out test genes.

Each node has exactly two input features: a constant equal to one and an indicator equal to one only for visible seed genes. Degree is not an explicit feature. Training uses a pairwise ranking loss between positive training targets and sampled genes that are not known disease genes.

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "bioGraph").is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from bioGraph.data.loading import (
    load_disease_genes,
    load_ppi_graph,
)
from bioGraph.gcn_prioritization.training import (
    predict_from_seed_genes,
    train_all_diseases,
    train_single_disease,
)
from bioGraph.evaluation.metrics import mean_reciprocal_rank_at_k, recall_at_k

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Load the biological graph and one disease

Node labels are Entrez gene IDs. Edges are undirected protein-protein interactions; the symbol is retained as a node attribute for readable output.

In [4]:
data_dir = project_root / "data" / "raw"
results_dir = project_root / "outputs" / "results"
reports_dir = project_root / "outputs" / "reports"
ppi_path = data_dir / "PPI202207.txt"
disease_path = data_dir / "pcbi.1004120.s004.txt"

graph = load_ppi_graph(data_dir / "PPI202207.txt")
diseases = load_disease_genes(data_dir / "pcbi.1004120.s004.txt")

disease_name = "breast neoplasms"
disease_genes = diseases[disease_name]

print(f"Graph: {graph.number_of_nodes():,} genes, {graph.number_of_edges():,} interactions")
print(f"{disease_name}: {len(disease_genes)} known genes before graph filtering")

Graph: 17,504 genes, 354,647 interactions
breast neoplasms: 40 known genes before graph filtering


## 2. Train

A shared outer split reserves 25% of the known genes for final testing. During training, fresh inner splits divide the outer training genes into visible seeds and positive labels. Held-out test genes never enter the input seed indicator, negative pool, or training loss. Final inference uses all outer training genes as visible seeds. Four calls to `torch.sparse.mm` propagate information over the full sparse PPI.

In [ ]:
result = train_single_disease(
    graph,
    disease_genes,
    hidden_dim=32,
    epochs=100,
    learning_rate=0.01,
    negative_ratio=5,
    train_fraction=0.75,
    inner_seed_fraction=2/3,
    seed=0,
)

print(
    f"Known genes in/outside graph: {result['known_in_graph']}/{result['known_not_in_graph']}\n"
    f"Outer split: {len(result['train_genes'])} training genes, "
    f"{len(result['test_genes'])} held-out tests\n"
    f"Ran {len(result['losses'])} epochs on {result['device']}; "
    f"final pairwise loss = {result['losses'][-1]:.4f}"
)

Known genes in/outside graph: 40/0
Split: 20 visible seeds, 10 positive training targets, 10 held-out tests
Ran 100 epochs on cpu; final pairwise loss = 0.5748


## 3. Evaluate and inspect candidates

Only visible seed genes are removed from the final candidate list. All remaining graph genes are ranked by descending logit. Recall@K measures the fraction of held-out test genes recovered in the top K, while MRR@K is the reciprocal rank of the first held-out test gene within K.

In [18]:
ranking = result["ranking"]
test_genes = result["test_genes"]

for k in (25, 100, 300):
    print(
        f"Recall@{k}: {recall_at_k(ranking, test_genes, k):.4f} | "
        f"MRR@{k}: {mean_reciprocal_rank_at_k(ranking, test_genes, k):.4f}"
    )

ranking[:10]

Recall@25: 0.0000 | MRR@25: 0.0000
Recall@100: 0.0000 | MRR@100: 0.0000
Recall@300: 0.1000 | MRR@300: 0.0055


[{'gene_id': 6714, 'symbol': 'SRC', 'score': 3.8197152614593506},
 {'gene_id': 7157, 'symbol': 'TP53', 'score': 3.7609829902648926},
 {'gene_id': 1457, 'symbol': 'CSNK2A1', 'score': 3.68658185005188},
 {'gene_id': 5566, 'symbol': 'PRKACA', 'score': 3.615516185760498},
 {'gene_id': 5594, 'symbol': 'MAPK1', 'score': 3.5541398525238037},
 {'gene_id': 983, 'symbol': 'CDK1', 'score': 3.5293805599212646},
 {'gene_id': 1956, 'symbol': 'EGFR', 'score': 3.5105743408203125},
 {'gene_id': 5578, 'symbol': 'PRKCA', 'score': 3.4427285194396973},
 {'gene_id': 1017, 'symbol': 'CDK2', 'score': 3.439793348312378},
 {'gene_id': 207, 'symbol': 'AKT1', 'score': 3.427990436553955}]

## 4. Train one shared GCN on all diseases

This trains **one model and one optimizer** across all disease tasks. A task consists of a disease-specific visible-seed indicator, positive training targets, and sampled negatives. Disease tasks are batched for efficiency: the graph and GCN weights are shared, while the second input feature changes with the disease. At inference time, provide a new seed set through that indicator and reuse the same trained model. The final table reports Recall@K and AP@K for each disease.

In [27]:
shared_gcn_result = train_all_diseases(
    graph,
    diseases,
    k_values=(25, 300),
    hidden_dim=32,
    epochs=100,
    learning_rate=0.01,
    weight_decay=1e-4,
    negative_ratio=5,
    train_fraction=0.75,
    inner_seed_fraction=2 / 3,
    seed=42,
    verbose=True,
    task_batch_size=8,
)

shared_model = shared_gcn_result["model"]
all_disease_results = shared_gcn_result["disease_results"]

Epoch   1/100: pairwise loss=0.6773
Epoch   2/100: pairwise loss=0.6559
Epoch   3/100: pairwise loss=0.6574
Epoch   4/100: pairwise loss=0.6579
Epoch   5/100: pairwise loss=0.6569
Epoch   6/100: pairwise loss=0.6567
Epoch   7/100: pairwise loss=0.6534
Epoch   8/100: pairwise loss=0.6466
Epoch   9/100: pairwise loss=0.6531
Epoch  10/100: pairwise loss=0.6575
Epoch  11/100: pairwise loss=0.6625
Epoch  12/100: pairwise loss=0.6463
Epoch  13/100: pairwise loss=0.6501
Epoch  14/100: pairwise loss=0.6470
Epoch  15/100: pairwise loss=0.6443
Epoch  16/100: pairwise loss=0.6436
Epoch  17/100: pairwise loss=0.6495
Epoch  18/100: pairwise loss=0.6436
Epoch  19/100: pairwise loss=0.6486
Epoch  20/100: pairwise loss=0.6419
Epoch  21/100: pairwise loss=0.6482
Epoch  22/100: pairwise loss=0.6403
Epoch  23/100: pairwise loss=0.6371
Epoch  24/100: pairwise loss=0.6380
Epoch  25/100: pairwise loss=0.6368
Epoch  26/100: pairwise loss=0.6273
Epoch  27/100: pairwise loss=0.6301
Epoch  28/100: pairwise loss

### Use the shared model for an input gene set

After training, inference needs only the shared model, prepared graph, and a set of visible seed genes. The returned ranking excludes those seeds.

In [28]:
query_seed_genes = diseases["breast neoplasms"][:10]
query_ranking = predict_from_seed_genes(
    shared_model,
    shared_gcn_result["graph_data"],
    query_seed_genes,
)
query_ranking[:20]

[{'gene_id': 7157, 'symbol': 'TP53', 'score': 4.406543731689453},
 {'gene_id': 5566, 'symbol': 'PRKACA', 'score': 4.026462078094482},
 {'gene_id': 4223, 'symbol': 'MEOX2', 'score': 3.904754400253296},
 {'gene_id': 207, 'symbol': 'AKT1', 'score': 3.8008694648742676},
 {'gene_id': 10155, 'symbol': 'TRIM28', 'score': 3.797651767730713},
 {'gene_id': 6714, 'symbol': 'SRC', 'score': 3.793501138687134},
 {'gene_id': 5578, 'symbol': 'PRKCA', 'score': 3.758572578430176},
 {'gene_id': 2932, 'symbol': 'GSK3B', 'score': 3.7151129245758057},
 {'gene_id': 1956, 'symbol': 'EGFR', 'score': 3.7061378955841064},
 {'gene_id': 1457, 'symbol': 'CSNK2A1', 'score': 3.65702223777771},
 {'gene_id': 5594, 'symbol': 'MAPK1', 'score': 3.5817887783050537},
 {'gene_id': 983, 'symbol': 'CDK1', 'score': 3.529280662536621},
 {'gene_id': 1499, 'symbol': 'CTNNB1', 'score': 3.509312391281128},
 {'gene_id': 5300, 'symbol': 'PIN1', 'score': 3.4893856048583984},
 {'gene_id': 5595, 'symbol': 'MAPK3', 'score': 3.480709314346

In [29]:
import pandas as pd

metric_columns = ["recall@25", "ap@25", "recall@300", "ap@300"]

missing_columns = [
    column
    for column in metric_columns
    if not all(
        column in disease_result["metrics"]
        for disease_result in all_disease_results.values()
    )
]
if missing_columns:
    raise RuntimeError(
        f"Missing metrics {missing_columns}. Rerun the shared GCN training cell "
        "to regenerate all_disease_results with AP values."
    )

metric_rows = [
    {
        "disease": disease_name,
        **{column: disease_result["metrics"][column] for column in metric_columns},
    }
    for disease_name, disease_result in all_disease_results.items()
]
metrics_by_disease = pd.DataFrame(metric_rows).set_index("disease")
display(metrics_by_disease)
display(metrics_by_disease.mean().rename("mean across diseases"))

,recall@25,ap@25,recall@300,ap@300
disease,,,,
adrenal gland diseases,0.000000,0.000000,0.000000,0.000000
alzheimer disease,0.000000,0.000000,0.125000,0.000427
amino acid metabolism inborn errors,0.000000,0.000000,0.230769,0.007501
amyotrophic lateral sclerosis,0.000000,0.000000,0.166667,0.000877
anemia aplastic,0.333333,0.044444,0.500000,0.056944
...,...,...,...,...
spondylarthropathies,0.200000,0.040000,0.200000,0.040000
tauopathies,0.000000,0.000000,0.125000,0.001923
uveal diseases,0.000000,0.000000,0.000000,0.000000


recall@25     0.053790
ap@25         0.010977
recall@300    0.195105
ap@300        0.014350
Name: mean across diseases, dtype: float64

In [30]:
breast_metrics = all_disease_results["breast neoplasms"]["metrics"]

breast_neoplasms_performance = pd.Series(
    {
        "Recall@25": breast_metrics["recall@25"],
        "AP@25": breast_metrics["ap@25"],
        "Recall@300": breast_metrics["recall@300"],
        "AP@300": breast_metrics["ap@300"],
    },
    name="breast neoplasms",
)
display(breast_neoplasms_performance)

Recall@25     0.100000
AP@25         0.010000
Recall@300    0.400000
AP@300        0.015439
Name: breast neoplasms, dtype: float64